# S01b — Ingestion dans le data lake : couche *bronze*

Objectif : déposer les données brutes, **sans les modifier**, dans le data lake S3 (RustFS),
au format Delta Lake, avec la trace de leur provenance.

```
data/raw (disque)  ──►  s3a://datalake/bronze/yellow_trips   (Delta)
                    ──►  s3a://datalake/bronze/zones          (Delta)
                    ──►  s3a://datalake/bronze/meteo          (Delta)
```

Console du data lake : http://localhost:9001 (rustfsadmin / rustfsadmin).

## 1. Créer le bucket

À faire une fois : dans la console RustFS (bouton *Create Bucket*, nom `datalake`), ou avec la cellule suivante.

In [ ]:
import sys; sys.path.append("/home/jovyan/work")
from pyspark.sql import functions as F
from commun.spark_utils import creer_session, RAW, LAKE

from commun.spark_utils import creer_bucket, format_tables
creer_bucket("datalake")

## 2. Session Spark branchée sur le data lake

Le premier lancement télécharge les jars Delta et S3 (plusieurs centaines de Mo) : à faire **avant** le cours, avec une bonne connexion.

In [ ]:
spark = creer_session("S01b")
FMT = format_tables(spark)   # 'delta' (ou 'parquet' si Delta indisponible)
FMT

## 3. Les courses → bronze

On ajoute deux colonnes techniques : le fichier d'origine et l'heure d'ingestion. Rien d'autre ne change.

In [ ]:
MOIS = "2025-01"
trips_raw = (spark.read.parquet(f"{RAW}/yellow_tripdata_{MOIS}.parquet")
             .withColumn("_source", F.lit(f"yellow_tripdata_{MOIS}.parquet"))
             .withColumn("_ingere_le", F.current_timestamp()))

(trips_raw.write.format(FMT).mode("overwrite")
          .save(f"{LAKE}/bronze/yellow_trips"))
print("bronze/yellow_trips :", spark.read.format(FMT).load(f"{LAKE}/bronze/yellow_trips").count(), "lignes")

Ouvrez la console RustFS : le dossier `bronze/yellow_trips` contient des fichiers Parquet **et** un dossier `_delta_log` (le journal des transactions Delta).

## 4. Référentiel des zones et météo → bronze

In [ ]:
zones = spark.read.csv(f"{RAW}/taxi_zone_lookup.csv", header=True, inferSchema=True)
zones.write.format(FMT).mode("overwrite").save(f"{LAKE}/bronze/zones")

# Le CSV Open-Meteo commence par 3 lignes de métadonnées : on le lit avec pandas
import pandas as pd
meteo_pd = pd.read_csv(f"{RAW}/meteo_nyc_{MOIS}.csv", skiprows=3)
meteo_pd.columns = ["horodatage", "temperature_c", "precipitation_mm"]
meteo = spark.createDataFrame(meteo_pd).withColumn("horodatage", F.to_timestamp("horodatage"))
meteo.write.format(FMT).mode("overwrite").save(f"{LAKE}/bronze/meteo")
meteo.show(3)

## 5. Pourquoi Delta Lake plutôt que du Parquet simple ?

Delta Lake apporte trois choses qu'un simple dossier de fichiers Parquet n'offre pas. Voyons-les une par une.

**a) Historique et *time travel*** : chaque écriture crée une version, qu'on peut relire.

In [ ]:
assert FMT == "delta", (
    "Delta Lake n'est pas actif (FMT = parquet) : regardez le message de creer_session() plus haut, "
    "puis Kernel → Restart et relancez le notebook depuis le début.")
print("Delta Lake actif : on peut faire les démonstrations.")

In [ ]:
# Une 2e écriture (ici : on ne garde que les courses payées par carte) crée la version 1
(spark.read.format("delta").load(f"{LAKE}/bronze/yellow_trips")
      .filter("payment_type = 1")
      .write.format("delta").mode("overwrite").save(f"{LAKE}/bronze/yellow_trips"))

try:
      from delta.tables import DeltaTable
except ModuleNotFoundError:
      DeltaTable = None
DeltaTable.forPath(spark, f"{LAKE}/bronze/yellow_trips").history() \
          .select("version", "timestamp", "operation").show(truncate=False)

In [ ]:
v0 = spark.read.format("delta").option("versionAsOf", 0).load(f"{LAKE}/bronze/yellow_trips").count()
v1 = spark.read.format("delta").load(f"{LAKE}/bronze/yellow_trips").count()
print(f"version 0 : {v0:,} lignes   |   version actuelle : {v1:,} lignes")

In [ ]:
# On restaure la version 0 : le bronze doit rester brut et complet
DeltaTable.forPath(spark, f"{LAKE}/bronze/yellow_trips").restoreToVersion(0)
print(spark.read.format("delta").load(f"{LAKE}/bronze/yellow_trips").count(), "lignes après restauration")

**b) Contrôle du schéma** : Delta refuse une écriture dont les colonnes ne correspondent pas.

In [ ]:
intrus = spark.createDataFrame([(1, "test")], ["VendorID", "colonne_inconnue"])
try:
    intrus.write.format("delta").mode("append").save(f"{LAKE}/bronze/yellow_trips")
except Exception as e:
    print("Écriture refusée :", str(e).splitlines()[0][:200])

**c) Évolutivité** : ajouter un mois = relancer le téléchargement puis la cellule 3
en mode `append`. Rien d'autre ne change dans la chaîne.

## C'est réussi quand…
- la console RustFS montre `bronze/yellow_trips`, `bronze/zones`, `bronze/meteo` ;
- l'historique Delta affiche au moins 3 versions et la table a retrouvé toutes ses lignes.

In [ ]:
spark.stop()